# Preprocessing Data Teks untuk Analisis Toxicity

Notebook ini berfokus pada tahap preprocessing data teks sebelum dilakukan pelatihan model. Tujuan utama dari proses ini adalah:
- membersihkan data yang tidak relevan atau tidak valid,
- mengurangi noise dan duplikat,
- menstandardisasi bentuk teks agar lebih konsisten,
- menyiapkan fitur teks yang siap dipakai oleh model pada tahap selanjutnya.

Pada bagian berikut, setiap sub-bagian menjelaskan alasan dan hasil dari proses preprocessing yang dilakukan secara berurutan.

In [1]:
import pandas as pd
df = pd.read_csv("../data/raw/indotoxic2024_annotated_data-3.csv")

In [2]:
# hapus spam
df = df[df["is_noise_or_spam_text"] == 0].copy()

# hapus teks yang kosong
empty_mask = (
    df["text"].isna() |
    df["text"].astype(str).str.strip().eq("")
)
df = df[~empty_mask].copy()

### 1. Duplicate Handling

Tujuan bagian ini adalah menghilangkan data yang berulang atau tidak konsisten agar model tidak belajar dari contoh yang sama berulang kali. Proses yang dilakukan meliputi:
- mendeteksi teks duplikat berdasarkan isi teks,
- menyimpan teks yang memiliki label konflik untuk dokumentasi,
- menghapus text yang menimbulkan ambiguitas label,
- menjaga satu representasi terbaik untuk setiap teks unik.

In [3]:
# Mengambil teks duplikat
duplicate_mask = df.duplicated(
    subset=["text"],
    keep=False
)

duplicates = df[duplicate_mask].copy()

# Mengambil teks yang memiliki label konflik
label_conflict = (
    df.groupby("text")["toxicity"]
      .nunique()
      .reset_index(name="n_label")
)

conflict_texts = label_conflict[
    label_conflict["n_label"] > 1
]["text"]

# simpan konflik ke file CSV utk dokumentasi
df_conflict = df[
    df["text"].isin(conflict_texts)
].copy()

df_conflict.to_csv(
    "../data/interim/label_conflict.csv",
    index=False
)

# hapus teks yang memiliki label konflik
df = df[
    ~df["text"].isin(conflict_texts)
].copy()

df = df.drop_duplicates(
    subset=["text"],
    keep="first"
).copy()

In [4]:
print("Shape akhir setelah cleaning:")
print(df.shape)

print("\nDuplicate text:")
print(df["text"].duplicated().sum())

Shape akhir setelah cleaning:
(23009, 17)

Duplicate text:
0


### 2. Ekstraksi Emoji

Bagian ini bertujuan untuk menangkap informasi tambahan dari teks yang sering mengandung ekspresi seperti emoji. Emoji dapat merepresentasikan emosi atau tone komentar, sehingga ekstraksi ini berguna untuk menjaga konteks semantik yang mungkin hilang saat teks dibersihkan. Hasil ekstraksi disimpan di kolom emoji untuk dipelajari lebih lanjut atau dianalisis sebagai fitur pendukung.

In [5]:
import emoji

def extract_emojis(text):
    return " ".join(
        item["emoji"]
        for item in emoji.emoji_list(str(text))
    )

df["emoji"] = df["text"].apply(extract_emojis)

In [6]:
df["emoji"].unique()

array(['', '😔', '👍', ..., '🔥 👍 🤑 💯 👍 🤑 🔥 💯 🤑 🔥 👍 💯', '🏨 🏨 🎀 📩 📩', '🇨🇳 🇲🇾'],
      shape=(2231,), dtype=object)

### 3. Cleaning Text

Tujuan bagian ini adalah membersihkan teks dari karakter noise yang tidak relevan untuk analisis sentimen atau toxicitas. Beberapa proses yang dilakukan meliputi:
- lowercase agar format teks konsisten,
- menghapus emoji, URL, HTML, mention, dan hashtag,
- menghilangkan tanda baca dan karakter berulang,
- menyederhanakan spasi agar teks menjadi lebih rapi dan standar.

Output dari proses ini adalah kolom text_clean yang siap diproses lebih lanjut.

In [7]:
import re

df["text_clean"] = df["text"].copy()

def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # Lowercase
    text = text.lower()
    
    # Hapus emoji
    text = emoji.replace_emoji(text, replace="")
    
    # Hapus URL
    text = re.sub(r"http\S+|www\S+", " ", text)
    
    # Hapus HTML tag seperti <br>, <br><br>
    text = re.sub(r"<[^>]+>", " ", text)
    
    # Hapus mention
    text = re.sub(r"@\w+", " ", text)
    
    # Hapus hashtag symbol tetapi pertahankan katanya
    # #SavePalestine -> SavePalestine
    text = re.sub(r"#(\w+)", r"\1", text)
    
    # Hilangkan punctuation
    text = re.sub(r"[^\w\s]", " ", text)
    
    # Normalisasi repeated characters
    text = re.sub(r"(.)\1{2,}", r"\1", text)
    
    # Normalisasi whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text
df["text_clean"] = df["text"].apply(clean_text)


### 4. Slang Normalization

Bagian ini bertujuan untuk menormalkan bahasa informal atau singkatan yang umum dipakai di media sosial. Banyak komentar mengandung kata slang, singkatan, atau variasi ejaan yang berbeda, sehingga perlu dikonversi ke bentuk baku agar model dapat memahami maknanya dengan lebih konsisten. Proses ini meningkatkan kualitas representasi teks sebelum masuk ke tahap pemodelan.

In [8]:
slang_dict = {
    # Negasi
    "gak": "tidak", "ga": "tidak", "gk": "tidak", "nggak": "tidak",
    "ngga": "tidak", "kaga": "tidak", "kagak": "tidak", "tdk": "tidak",

    # Kata ganti & partikel umum
    "yg": "yang", "dgn": "dengan", "bgt": "banget", "aja": "saja",
    "sy": "saya", "gw": "saya", "gue": "saya", "gua": "saya",
    "lu": "kamu", "lo": "kamu", "elo": "kamu", "km": "kamu",
    "kmu": "kamu", "elu": "kamu", "w": "saya",

    # Kata tanya & keterangan
    "krn": "karena", "krna": "karena", "karna": "karena",
    "utk": "untuk", "buat": "untuk", "jd": "jadi", "jgn": "jangan",
    "jgnkan": "jangankan", "gmn": "bagaimana", "gimana": "bagaimana",
    "knp": "kenapa", "kenapa": "mengapa", "dmn": "dimana",
    "drmn": "dari mana", "kpn": "kapan", "brp": "berapa",
    "brapa": "berapa", "sm": "sama", "sma": "sama",

    # Waktu
    "skrg": "sekarang", "skrng": "sekarang", "td": "tadi",
    "bsk": "besok", "kmrn": "kemarin", "kmaren": "kemarin",
    "tar": "nanti", "ntar": "nanti", "nti": "nanti",

    # Kata kerja & sifat
    "udh": "sudah", "udah": "sudah", "dah": "sudah", "blm": "belum",
    "blum": "belum", "bkn": "bukan", "bnr": "benar", "bener": "benar",
    "emg": "memang", "emang": "memang", "tau": "tahu", "gatau": "tidak tahu",
    "gtau": "tidak tahu", "pgn": "ingin", "pengen": "ingin",
    "pngen": "ingin", "mau": "ingin", "bs": "bisa", "bsa": "bisa",
    "hrs": "harus", "harusnya": "seharusnya",

    # Kata sifat & ekspresi
    "bgt": "banget", "bngt": "banget", "bener2": "benar-benar",
    "cape": "capai", "capek": "lelah", "cakep": "cantik/tampan", 
    "kece": "keren", "mantul": "mantap betul", "mantep": "mantap",
    "gokil": "gila", "parah": "sangat", "anjay": "ekspresi kaget/kagum",
    "santuy": "santai", "woles": "santai", "baper": "bawa perasaan",
    "kepo": "ingin tahu urusan orang", "julid": "iri/sirik",
    "ambyar": "hancur/berantakan", "receh": "lucu remeh",

    # Singkatan chat umum
    "otw": "on the way / dalam perjalanan", "gpp": "tidak apa-apa",
    "gapapa": "tidak apa-apa", "cmiiw": "correct me if I'm wrong",
    "btw": "ngomong-ngomong", "fyi": "sebagai informasi",
    "pls": "tolong", "thx": "terima kasih", "makasih": "terima kasih",
    "gaje": "tidak jelas", "japri": "jalur pribadi (chat pribadi)",
    "japrii": "jalur pribadi", "wkwk": "tertawa", "wkwkwk": "tertawa",
    "haha": "tertawa", "anjir": "ekspresi kaget", "yaudah": "ya sudah",
    "udahlah": "sudahlah", "kayanya": "kayaknya", "kayaknya": "sepertinya",
    "kyk": "seperti", "kek": "seperti", "sbnrnya": "sebenarnya",
    "sebenarnya": "pada dasarnya", "org": "orang", "org2": "orang-orang",
    "tp": "tapi", "tpi": "tapi", "dr": "dari", "trs": "terus",
    "trus": "terus", "abis": "habis", "abisin": "habiskan",
    "duit": "uang", "cuan": "keuntungan/uang", "gaje": "tidak jelas",
}

In [9]:
def normalize_repeated_char(text):
    text = re.sub(r"(.)\1{2,}", r"\1", text)
    return text

def normalize_slang(text):
    words = text.split()
    
    normalized = [
        slang_dict.get(word, word)
        for word in words
    ]
    
    return " ".join(normalized)

df["text_clean"] = df["text_clean"].apply(normalize_repeated_char)
df["text_clean"] = df["text_clean"].apply(normalize_slang)

### 5. Topic Handling

Tujuan dari bagian ini adalah menyiapkan struktur data untuk kolom topic agar lebih mudah diproses dan divisualisasikan. Teks topic biasanya berbentuk string yang berisi beberapa kategori dengan pemisah koma, sehingga perlu diubah menjadi list agar lebih efektif untuk analisis lebih lanjut, seperti pemetaan tema atau eksplorasi label.

In [10]:
df["topic_list"] = (
    df["topic"]
    .fillna("")
    .apply(
        lambda x: [
            topic.strip()
            for topic in str(x).split(",")
            if topic.strip()
        ]
    )
)

In [11]:
df[["topic", "topic_list"]].head()

,topic,topic_list
0,Disabilitas,[Disabilitas]
1,Jewish,[Jewish]
2,"Terpolarisasi, Tionghoa","[Terpolarisasi, Tionghoa]"
3,"Terpolarisasi, Jewish","[Terpolarisasi, Jewish]"
6,"Terpolarisasi, Tionghoa","[Terpolarisasi, Tionghoa]"


### Stopword

In [12]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stopword_factory = StopWordRemoverFactory()
stopwords = set(stopword_factory.get_stop_words())

In [13]:
def remove_stopwords(text, stopwords=stopwords):
    if pd.isna(text):
        return ""
    
    words = str(text).split()
    
    filtered_words = [
        word for word in words
        if word not in stopwords
    ]
    
    return " ".join(filtered_words)

### Stemming

In [14]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.preprocessing import MultiLabelBinarizer

In [15]:
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

In [16]:
def stem_text(text):
    if pd.isna(text):
        return ""
    
    return stemmer.stem(str(text))

### Label Encoder

In [17]:
def encode_multilabel_column(data, column):
    """
    Mengubah kolom yang berisi list label menjadi multi-hot encoding.
    """
    
    mlb = MultiLabelBinarizer()
    
    encoded = mlb.fit_transform(data[column])
    
    encoded_df = pd.DataFrame(
        encoded,
        columns=[
            f"{column}_{label}"
            for label in mlb.classes_
        ],
        index=data.index
    )
    
    return encoded_df, mlb

In [18]:
def re_preprocess(
    data,
    stopword_remove=True,
    encode_topic=False,
    stem=True
):
    
    df = data.copy()
    
    # =========================
    # 1. Pastikan text_clean ada
    # =========================
    if "text_clean" not in df.columns:
        raise ValueError(
            "Kolom 'text_clean' tidak ditemukan."
        )
    
    # =========================
    # 2. Stopword Removal
    # =========================
    if stopword_remove:
        df["text_processed"] = df["text_clean"].apply(
            remove_stopwords
        )
    else:
        df["text_processed"] = df["text_clean"]
    
    # =========================
    # 3. Stemming
    # =========================
    if stem:
        df["text_processed"] = df["text_processed"].apply(
            stem_text
        )
    
    # =========================
    # 4. Topic normalization
    # =========================
    if "topic" in df.columns:
        df["topic_list"] = df["topic"].apply(
            normalize_topic_list
        )
    
    # =========================
    # 5. Label Encoding
    # =========================
    topic_encoder = None
    
    if LE and "topic_list" in df.columns:
        
        topic_encoded, topic_encoder = encode_multilabel_column(
            df,
            "topic_list"
        )
        
        df = pd.concat(
            [df, topic_encoded],
            axis=1
        )
    
    return df, topic_encoder

In [ ]:
df_processed, topic_encoder = re_preprocess(
    df,
    stopword_remove=True,
    encode_topic=False,
    stem=True
)

### 6. Cek Hasil Akhir

Bagian ini berfungsi sebagai validasi kualitas preprocessing. Tujuannya adalah memastikan tidak ada teks kosong yang tersisa setelah proses pembersihan dan bahwa data yang sudah dibersihkan siap digunakan pada tahapan berikutnya. Proses pengecekan ini penting agar hasil akhir konsisten dan meminimalkan error dalam pelatihan model.

In [ ]:
empty_after_cleaning = (
    df["text_clean"].str.strip().eq("")
)

print(
    "Text kosong setelah cleaning:",
    empty_after_cleaning.sum()
)
df = df[~empty_after_cleaning].copy()

Text kosong setelah cleaning: 0


In [ ]:
pd.set_option("display.max_colwidth", 200)

df[
    ["text", "emoji", "text_clean", "topic_list"]
].sample(10)

,text,emoji,text_clean,topic_list
18887,Bisa Kalii Ni Messi Abis Dapat Ballon D'or Jadi Duta Perdamaian Palestina - Israel,,bisa kalii ni messi habis dapat ballon d or jadi duta perdamaian palestina israel,[Jewish]
7513,Rasanya kayak mau gila,,rasanya kayak ingin gila,[Disabilitas]
34862,@AnakBawang_0705 @F_Gavriel_Joko @Ir_vnd @KUda_niell @AliyasKartino Santai aja org Kristen gak cengeng dikit2 ngamuk teriak2 PENIS taan 🤣🤣🤣🤣🤣🤣🤣 https://t.co/b9DKANc1Zj,🤣 🤣 🤣 🤣 🤣 🤣 🤣,santai saja orang kristen tidak cengeng dikit2 ngamuk teriak2 penis taan,[Kristen]
17394,"Renungan Harian Penyuluh Agama Katolik Injil (04/09), menceritakan Yesus mengalami penolakan di kampung halaman-Nya sendiri. Ketika dihadapkan pada peristiwa ditolak, diremehkan oleh orang lain, m...",,renungan harian penyuluh agama katolik injil 04 09 menceritakan yesus mengalami penolakan di kampung halaman nya sendiri ketika dihadapkan pada peristiwa ditolak diremehkan oleh orang lain maka sa...,[Kristen]
10420,dan Jokowow akhirnya bisa mengurangi angka pengangguran di China #Bercanda,,dan jokowow akhirnya bisa mengurangi angka pengangguran di china bercanda,[Tionghoa]
427,Politik Ganjar Mahfud 💞 Politik yg baik bkn yg licik. Politik yg ikhlas bkn yg culas. Politik kebangsaan bkn kebangsatan. Politik beradab bkn biadab. Politik yg mengantarkan kesejahteraan bkn yg h...,💞,politik ganjar mahfud politik yang baik bukan yang licik politik yang ikhlas bukan yang culas politik kebangsaan bukan kebangsatan politik beradab bukan biadab politik yang mengantarkan kesejahter...,"[Disabilitas, Terpolarisasi]"
17119,"UNIVERSITAS KATOLIK PARAHYANGAN LIFE UNPAR Akhir minggu pergi ke Danau Toba, tak lupa bawa buku bacaan\nNikmatnya hari Jumat telah tiba, selamat berakhir pekan Unparian!\n\nLepaskan penat kalian, ...",,universitas katolik parahyangan life unpar akhir minggu pergi ke danau toba tak lupa bawa buku bacaan nikmatnya hari jumat telah tiba selamat berakhir pekan unparian lepaskan penat kalian menghabi...,[Kristen]
8344,Sinting cape bgt hr ini 😵‍💫,😵‍💫,sinting capai banget hr ini,[Disabilitas]
11820,"Indonesia dikalahkan China 0-3 dalam babak perempat final beregu Asian Games 2022, setelah Putri Kusuma Wardani kalah dari lawannya dengan skor 15-21, 19-21. #AsianGames2022 📸: ANTARA FOTO/M Risya...",📸,indonesia dikalahkan china 0 3 dalam babak perempat final beregu asian games 2022 setelah putri kusuma wardani kalah dari lawannya dengan skor 15 21 19 21 asiangames2022 antara foto m risyal hiday...,[Tionghoa]
9067,FajRi gagal lagi di babak pertama. Kali ini di China Open. Peringkat 1 Dunia yang konsisten untuk tidak konsisten. 😌 #ChinaOpenSuper1000,😌,fajri gagal lagi di babak pertama kali ini di china open peringkat 1 dunia yang konsisten untuk tidak konsisten chinaopensuper10,[Tionghoa]


In [ ]:
df.to_csv("../data/processed/data_cleaned_before_encode.csv")